# ObjectLab: ArtAug photo edits in Kaggle

The embedded input bundle contains ten candidate training photos and four prompts. This hand-in includes generation sections for the first four photos only: one bottle, cup, pen, and pair of scissors. The executed notebook preserves the four generated results; photos 5–10 were not generated and are omitted from this notebook.

1. In Kaggle, open **Settings / Session options**, choose **Accelerator → GPU**, and enable **Internet**. If two GPUs are assigned, the notebook uses the first. Keep the working notebook private.
2. To download gated model files automatically, open **Add-ons → Secrets**. Add a secret named `HF_TOKEN` containing a Hugging Face read token, then enable it for this notebook. The same account must have access to both models: [FLUX.1-Kontext-dev](https://huggingface.co/black-forest-labs/FLUX.1-Kontext-dev) and [FLUX.1-dev](https://huggingface.co/black-forest-labs/FLUX.1-dev).
3. Run the setup cells in order, then run **Photo 1 of 4** through **Photo 4 of 4**. Each photo cell starts generation when you run it.

If the complete weights are already attached to the Kaggle session, Hugging Face access is not needed for those files. Internet access is still needed to install Python packages. See [Kaggle Notebooks](https://www.kaggle.com/docs/notebooks) and the [ArtAug adapter](https://huggingface.co/DiffSynth-Studio/FLUX.1-Kontext-dev-lora-ArtAug).

## 1. Check available resources

Run the next cell to see available disk space, system memory, and memory on each detected GPU. A fresh model-weight download needs about **34.9 GiB** of temporary disk space including reserve. RAM and GPU memory are checked separately.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import base64, csv, hashlib, io, json, os, shutil, subprocess, sys, tempfile, time, uuid, zipfile
import importlib.metadata, platform

KAGGLE_WORKING = Path('/kaggle/working')
KAGGLE_INPUT = Path('/kaggle/input')
if not KAGGLE_WORKING.is_dir():
    raise RuntimeError('Откройте этот .ipynb в редакторе Kaggle и запустите сессию с GPU.')
PACKAGE_DIR = KAGGLE_WORKING/'ObjectLab_ArtAug_10'
PACKAGE_DIR.mkdir(parents=True, exist_ok=True)
INPUT_DIR = PACKAGE_DIR/'images'
OUTPUT_DIR = PACKAGE_DIR/'artaug_pilot'
DATASET_MANIFEST_PATH = PACKAGE_DIR/'dataset_manifest.csv'
SELECTION_PATH = PACKAGE_DIR/'pilot_selection.json'
EXPORT_PATH = KAGGLE_WORKING/'ArtAug_Pilot_Output.zip'
STATUS_PATH = PACKAGE_DIR/'artaug_run_status.json'
RUN_STATUS = {'runtime': 'Kaggle', 'status': 'started', 'stage': 'setup', 'errors': [],
              'python': sys.version.split()[0], 'started_utc': datetime.now(timezone.utc).isoformat()}

def write_run_status():
    RUN_STATUS['updated_utc'] = datetime.now(timezone.utc).isoformat()
    tmp = STATUS_PATH.with_name(STATUS_PATH.name + '.' + uuid.uuid4().hex + '.tmp')
    try:
        tmp.write_text(json.dumps(RUN_STATUS, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
        os.replace(tmp, STATUS_PATH)
    finally:
        tmp.unlink(missing_ok=True)

def session_ram():
    import psutil
    host = psutil.virtual_memory()
    total, free = host.total, host.available
    for directory, limit_name, usage_name in [
        (Path('/sys/fs/cgroup'), 'memory.max', 'memory.current'),
        (Path('/sys/fs/cgroup/memory'), 'memory.limit_in_bytes', 'memory.usage_in_bytes')
    ]:
        try:
            limit = int((directory/limit_name).read_text().strip())
            used = int((directory/usage_name).read_text().strip())
        except (OSError, ValueError):
            continue
        if 0 < limit < total:
            try:
                values = dict(line.split() for line in (directory/'memory.stat').read_text().splitlines())
                reclaimable = int(values.get('total_inactive_file', values.get('inactive_file', 0)))
            except (OSError, ValueError):
                reclaimable = 0
            total, free = limit, max(0, min(free, limit, limit - used + reclaimable))
    return total, free

total_ram, free_ram = session_ram()
print(f'Оперативная память · сейчас доступно {free_ram/2**30:.1f} из {total_ram/2**30:.1f} GiB.')
for location in [Path('/kaggle/temp'), Path('/tmp'), KAGGLE_WORKING]:
    if location.is_dir():
        usage = shutil.disk_usage(location)
        print(f'Диск {location} · сейчас свободно {usage.free/2**30:.1f} GiB.')
print('Файлы одной файловой системы могут отображаться под разными путями.')
write_run_status()
try:
    gpu = subprocess.run(['nvidia-smi', '--query-gpu=index,name,memory.free,memory.total',
                          '--format=csv,noheader,nounits'], text=True, capture_output=True, timeout=15)
    if gpu.returncode or not gpu.stdout.strip():
        raise OSError()
except (OSError, subprocess.TimeoutExpired):
    raise RuntimeError('GPU не обнаружен. В Session options → Accelerator выберите GPU, дождитесь запуска и повторите эту ячейку.') from None
for line in gpu.stdout.strip().splitlines():
    index, name, free, total = (x.strip() for x in line.split(','))
    print(f'GPU {index} · {name} · свободно {int(free)/1024:.1f} из {int(total)/1024:.1f} GiB видеопамяти.')
print('Обработка использует одну видеокарту (GPU 0). ZIP появится в', EXPORT_PATH)

## 2. Prepare the photo bundle and restore an earlier result

Run the cells in this section once per session. The notebook contains an embedded input bundle with the ten candidate training photos. If you have an earlier **ArtAug_Pilot_Output.zip**, attach it through **Add Input** as the output of the previous notebook run. You may also attach a private dataset ZIP. Kaggle may unpack uploaded archives, so the loader supports both the ZIP file and an extracted `artaug_pilot` folder. The input bundle and output ZIP are different: the input bundle contains candidate source photos; the output ZIP contains generated, reviewed pairs.

In [ ]:
INPUT_BUNDLE_B64 = (
    "..."
)
packed = base64.b64decode(INPUT_BUNDLE_B64, validate=True)
if hashlib.sha256(packed).hexdigest() != '1f3a0127fa64803f15a946dbd65e5d956190d41facfeabb3ccf7fbf73364a261':
    raise RuntimeError('Встроенный пакет повреждён. Заново откройте скачанный .ipynb.')
expected_names = ['pilot_selection.json', 'dataset_manifest.csv', 'artaug_weights/manifest.json', 'images/train/bottles/photo_5426963673156099989_y.jpg', 'images/train/cups/6.jpg', 'images/train/pens/78.jpg', 'images/train/scissors/62.jpg', 'images/train/bottles/89.jpg', 'images/train/cups/50.jpg', 'images/train/pens/66.jpg', 'images/train/scissors/photo_5426963673156100148_y.jpg', 'images/train/bottles/75.jpg', 'images/train/pens/91.jpg']
with zipfile.ZipFile(io.BytesIO(packed)) as bundle:
    if sorted(bundle.namelist()) != sorted(expected_names): raise ValueError('Неожиданный состав входов.')
    for name in expected_names:
        destination = PACKAGE_DIR/name
        destination.parent.mkdir(parents=True,exist_ok=True)
        destination.write_bytes(bundle.read(name))
del packed
print('Исходники извлечены в', INPUT_DIR)

In [ ]:
CLASS_ORDER = ["bottles", "cups", "pens", "scissors"]
PILOT_SELECTION = json.loads(SELECTION_PATH.read_text(encoding="utf-8"))
SOURCE_MANIFEST_SHA256 = PILOT_SELECTION.get("source_manifest_sha256")
if PILOT_SELECTION.get("schema_version") != 1:
    raise RuntimeError("pilot_selection.json: ожидалась schema_version=1")
if not isinstance(SOURCE_MANIFEST_SHA256, str) or len(SOURCE_MANIFEST_SHA256) != 64:
    raise RuntimeError("pilot_selection.json не содержит SHA-256 dataset_manifest.csv")

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

if sha256_file(DATASET_MANIFEST_PATH) != SOURCE_MANIFEST_SHA256:
    raise RuntimeError("SHA-256 dataset_manifest.csv не совпадает с замороженным pilot_selection.json")
with DATASET_MANIFEST_PATH.open("r", encoding="utf-8-sig", newline="") as stream:
    dataset_rows = list(csv.DictReader(stream))
dataset_by_path = {row["path"].replace("\\", "/"): row for row in dataset_rows}
SELECTED = PILOT_SELECTION.get("photos")
if not isinstance(SELECTED, list) or len(SELECTED) != 10:
    raise RuntimeError("pilot_selection.json должен содержать ровно 10 записей в photos")
seen_paths = set()
for index, row in enumerate(SELECTED):
    rel = row.get("path", "")
    parts = Path(rel.replace("\\", "/")).parts
    if not rel or Path(rel).is_absolute() or ".." in parts or "\\" in rel or ":" in rel:
        raise RuntimeError(f"Небезопасный путь в записи {index}: {rel!r}")
    rel = "/".join(parts)
    if rel in seen_paths:
        raise RuntimeError(f"Повтор исходной фотографии: {rel}")
    seen_paths.add(rel)
    if len(parts) < 3 or parts[0] != "train" or parts[1] not in CLASS_ORDER:
        raise RuntimeError(f"Допускаются только train/{CLASS_ORDER} : {rel}")
    if row.get("class_name") != parts[1]:
        raise RuntimeError(f"Метка класса не совпадает с папкой: {rel}")
    catalog = dataset_by_path.get(rel)
    if catalog is None or catalog.get("split") != "train" or catalog.get("class") != row["class_name"] or catalog.get("sha256") != row.get("sha256"):
        raise RuntimeError(f"Выбор не подтверждён замороженным dataset_manifest.csv: {rel}")
    source_path = INPUT_DIR.joinpath(*parts)
    if not source_path.is_file():
        raise FileNotFoundError(f"Нет исходной фотографии с исходным именем: {source_path}")
    if sha256_file(source_path) != row["sha256"]:
        raise RuntimeError(f"SHA-256 исходной фотографии изменился: {rel}")

if len({row["class_name"] for row in SELECTED}) != 4:
    raise RuntimeError("Нужно хотя бы по одной тренировочной фотографии каждого класса")
print("Проверено 10/10 исходных train-фотографий; SHA-256 таблицы и файлов совпадают.")
for i, row in enumerate(SELECTED, 1):
    print(f"{i:02d}. {row['class_name']:8s} {row['path']}")

In [ ]:
from IPython.display import Image as DisplayImage, display
for i, row in enumerate(SELECTED, 1):
    path = INPUT_DIR.joinpath(*Path(row["path"]).parts)
    display(DisplayImage(filename=str(path), width=420))
    print(f"{i}. {row['class_name']} — {Path(row['path']).name}")

In [ ]:
def _check_manifest(data):
    if data.get('schema_version') != 1 or data.get('source_manifest_sha256') != SOURCE_MANIFEST_SHA256:
        raise ValueError('Архив относится к другой выборке.')
    generator = data.get('generator') or {}
    weights_spec = json.loads((PACKAGE_DIR/'artaug_weights/manifest.json').read_text())
    for key, name, revision in [('kontext','base_model_id','base_revision'),
                                ('flux_dev','aux_model_id','aux_revision'),
                                ('artaug','adapter_id','adapter_revision')]:
        expected = weights_spec['repos'][key]
        if (generator.get(name),generator.get(revision)) != (expected['repo_id'],expected['revision']):
            raise ValueError('Архив относится к другому генератору или версии.')
    if generator.get('synthetic_images') is not True:
        raise ValueError('В архиве отсутствует метка синтетического результата.')
    examples = data.get('examples')
    if not isinstance(examples,list) or not 1 <= len(examples) <= 10:
        raise ValueError('Ожидается от 1 до 10 готовых пар.')
    by_source = {row['path']:(i,row) for i,row in enumerate(SELECTED)}
    seen = set()
    for row in examples:
        i, src = by_source[row['source_manifest_path']]
        image_id = hashlib.sha256((src['class_name']+'\n'+src['path']).encode()).hexdigest()[:16]
        if image_id in seen or row.get('id') != image_id or row.get('class_name') != src['class_name']:
            raise ValueError('Некорректный ID или класс в архиве.')
        seen.add(image_id)
        if row.get('source_sha256') != src['sha256'] or row.get('original_sha256') != src['sha256']:
            raise ValueError('Исходник в архиве отличается от проверенной тренировочной фотографии.')
        if row.get('seed') != 4127+i or not row.get('prompt') or row.get('approved') is not False:
            raise ValueError('Проверьте промпт, seed и статус проверки в архиве.')
        for kind in ('original','enhanced'):
            if row.get(kind) != f'images/{image_id}_{kind}.jpg':
                raise ValueError('Путь к изображению отличается от ожидаемого.')
    return {row['id']:row for row in examples}

def _verified_zip(path):
    # Only allow the small manifest and the specific JPEGs named by it.
    with zipfile.ZipFile(path) as archive:
        info = archive.infolist()
        if (len(info) > 22 or len(info) != len({entry.filename for entry in info}) or
                sum(entry.file_size for entry in info) > 160*1024**2):
            raise ValueError('Некорректный размер или состав архива результатов.')
        manifest = json.loads(archive.read('artaug_pilot/manifest.json'))
        rows = _check_manifest(manifest)
        payload = {}
        allowed = {'artaug_pilot/manifest.json','artaug_pilot/run_status.json'}
        for row in rows.values():
            for kind in ('original','enhanced'):
                name = 'artaug_pilot/'+row[kind]
                allowed.add(name)
                image = archive.read(name)
                if not image.startswith(b'\xff\xd8\xff') or hashlib.sha256(image).hexdigest() != row[kind+'_sha256']:
                    raise ValueError(f'Повреждено изображение в архиве: {name}')
                payload[row[kind]] = image
        if set(archive.namelist()) - allowed:
            raise ValueError('В архиве найдены посторонние файлы.')
    return manifest, rows, payload

def _atomic_bytes(path, data):
    path.parent.mkdir(parents=True,exist_ok=True)
    tmp = path.with_name(path.name+'.tmp-'+uuid.uuid4().hex)
    try:
        tmp.write_bytes(data)
        os.replace(tmp,path)
    finally:
        tmp.unlink(missing_ok=True)

def restore_checkpoint(path):
    if path is None or not path.is_file():
        return
    _restore_pairs(*_verified_zip(path))

def _restore_pairs(imported, remote, payload):
    target = OUTPUT_DIR/'manifest.json'
    existing = json.loads(target.read_text()) if target.exists() else None
    if existing is not None:
        current = _check_manifest(existing)
        if existing['generator'] != imported['generator']:
            raise ValueError('В рабочей папке и архиве разные параметры модели. Существующие результаты сохранены.')
        if any(current[k] != remote[k] for k in current.keys() & remote.keys()):
            raise ValueError('Две разные пары с одинаковым ID. Существующие результаты сохранены.')
        for row in current.values():
            for kind in ('original','enhanced'):
                path = OUTPUT_DIR/row[kind]
                if not path.is_file() or sha256_file(path) != row[kind+'_sha256']:
                    raise ValueError('Проверьте уже сохранённый локальный файл: '+str(path))
    else:
        current = {}
    for rel,data in payload.items():
        path = OUTPUT_DIR/rel
        if path.exists() and hashlib.sha256(path.read_bytes()).digest() != hashlib.sha256(data).digest():
            raise ValueError('На диске среды найден другой результат: '+rel)
    for rel,data in payload.items():
        path = OUTPUT_DIR/rel
        if not path.exists(): _atomic_bytes(path,data)
    merged = {**remote,**current}
    imported['examples'] = list(merged.values())
    _atomic_bytes(target,(json.dumps(imported,ensure_ascii=False,indent=2)+'\n').encode())
    print(f'Восстановлено готовых пар: {len(merged)}/10. Их можно открыть повторно без генерации.')


def recover_input_folder(folder):
    manifest_file = folder/'manifest.json'
    if manifest_file.stat().st_size > 1024**2:
        raise ValueError('Слишком большой манифест результатов.')
    manifest = json.loads(manifest_file.read_text(encoding='utf-8'))
    rows = _check_manifest(manifest)
    names = ['manifest.json'] + [row[kind] for row in rows.values() for kind in ('original', 'enhanced')]
    if sum((folder/name).stat().st_size for name in names) > 160*1024**2:
        raise ValueError('Слишком большой комплект изображений.')
    payload = {}
    for row in rows.values():
        for kind in ('original', 'enhanced'):
            data = (folder/row[kind]).read_bytes()
            if not data.startswith(b'\xff\xd8\xff') or hashlib.sha256(data).hexdigest() != row[kind+'_sha256']:
                raise ValueError('Повреждён JPEG в подключённых результатах: '+row[kind])
            payload[row[kind]] = data
    _restore_pairs(manifest, rows, payload)

if EXPORT_PATH.is_file():
    restore_checkpoint(EXPORT_PATH)
if KAGGLE_INPUT.is_dir():
    for parent, dirs, files in os.walk(KAGGLE_INPUT):
        folder = Path(parent)
        dirs[:] = [d for d in dirs if d not in {'images', '.cache', 'artaug_weights', 'kontext', 'flux_dev', 'artaug'}]
        if 'ArtAug_Pilot_Output.zip' in files:
            restore_checkpoint(folder/'ArtAug_Pilot_Output.zip')
        if folder.name == 'artaug_pilot' and 'manifest.json' in files:
            recover_input_folder(folder)
            dirs.clear()
print('Можно переходить к выбору хранилища весов.')

## 3. Choose where to store model weights

By default, Kaggle downloads **14 files totaling about 34.22 GiB**. The next cell checks free space and reuses files already in the cache. Temporary files may be removed when the session ends.

If the temporary disk is too small, use an existing `artaug_weights` folder. In Kaggle, create a **Private** dataset with the `kontext`, `flux_dev`, and `artaug` subfolders, then attach it through **Add Input → Your Datasets**. Verify that the nested folders were preserved and run the next cell again. The notebook will read those files from the attached input. See [Kaggle Datasets](https://www.kaggle.com/docs/datasets).

In [ ]:
PREFLIGHT_OK = False
from pathlib import Path
import hashlib, json, os, shutil, tempfile

assert "PACKAGE_DIR" in globals(), "Сначала распакуйте встроенный пакет."
PACKAGE_DIR = Path(PACKAGE_DIR).resolve()
WEIGHTS_MANIFEST_PATH = PACKAGE_DIR / "artaug_weights" / "manifest.json"
PINNED_MANIFEST_SHA256 = "3f988847f8f6c17741b261b4ae2c32d419230831f399c6a9eda082b8470dd8e1"
if not WEIGHTS_MANIFEST_PATH.is_file():
    raise FileNotFoundError(f"Не найден встроенный манифест весов: {WEIGHTS_MANIFEST_PATH}")
_manifest_bytes = WEIGHTS_MANIFEST_PATH.read_bytes()
if hashlib.sha256(_manifest_bytes).hexdigest() != PINNED_MANIFEST_SHA256:
    raise RuntimeError("Встроенный манифест весов не совпадает с закреплённой версией ноутбука.")
WEIGHTS_MANIFEST = json.loads(_manifest_bytes.decode("utf-8"))
if WEIGHTS_MANIFEST.get("schema_version") != 1 or len(WEIGHTS_MANIFEST.get("files", {})) != 14:
    raise RuntimeError("Ожидался закреплённый манифест с 14 файлами версии schema 1.")
WEIGHT_RECORDS = WEIGHTS_MANIFEST["files"]
WEIGHTS_READY_FOR_LOCAL_LOAD = False
HF_WEIGHT_ACCESS_VERIFIED = False
TOTAL_WEIGHT_BYTES = sum(int(x["size_bytes"]) for x in WEIGHT_RECORDS.values())

def _files_root(_path):
    """Accept a weights folder, a dataset root, or a wrapper around either."""
    _path = Path(_path)
    for _candidate in (_path, _path / "artaug_weights"):
        if all((_candidate / _group).is_dir() for _group in ("kontext", "flux_dev", "artaug")):
            return _candidate
    return None

def _complete(_root):
    try:
        _root = Path(_root)
        # Attached manifests are optional, but if supplied they must be the exact
        # pinned manifest. The authoritative spec always comes from this notebook.
        _mp = _root / "manifest.json"
        if _mp.is_file() and hashlib.sha256(_mp.read_bytes()).hexdigest() != PINNED_MANIFEST_SHA256:
            return False
        for _rel, _rec in WEIGHT_RECORDS.items():
            _p = _root.joinpath(*_rel.split("/"))
            if not _p.is_file() or _p.stat().st_size != int(_rec["size_bytes"]):
                return False
        return True
    except (OSError, ValueError, TypeError):
        return False

# Kaggle uses both /kaggle/input/<slug>/... and the newer
# /kaggle/input/datasets/<owner>/<slug>/versions/<n>/... layout.
_input_roots = []
for _raw in (globals().get("KAGGLE_INPUT"), os.environ.get("KAGGLE_INPUT"), "/kaggle/input"):
    if _raw:
        _p = Path(_raw)
        if _p.is_dir() and _p not in _input_roots:
            _input_roots.append(_p)
_input_candidates = []
for _root in _input_roots:
    _paths = [_root]
    _paths += list(_root.glob("*"))
    _paths += list(_root.glob("datasets/*/*"))
    _paths += list(_root.glob("datasets/*/*/versions/*"))
    # Also allow KAGGLE_INPUT itself to point at the datasets hierarchy.
    _paths += list(_root.glob("*/*"))
    _paths += list(_root.glob("*/*/versions/*"))
    for _p in _paths:
        _found = _files_root(_p)
        if _found is not None and _found not in _input_candidates:
            _input_candidates.append(_found)
        # A wrapper directory may contain a nested artaug_weights or groups.
        _found = _files_root(_p / "artaug_weights") if (_p / "artaug_weights").is_dir() else None
        if _found is not None and _found not in _input_candidates:
            _input_candidates.append(_found)

_complete_input = next((p for p in _input_candidates if _complete(p)), None)
if _complete_input is not None:
    LOCAL_WEIGHTS_DIR = _complete_input
    WEIGHT_SOURCE = "kaggle_input"
    WEIGHTS_READY_FOR_LOCAL_LOAD = True
    print("Найден полный набор весов в Kaggle Input:", LOCAL_WEIGHTS_DIR)
    print("Загрузчик дополнительно проверит контрольные суммы всех файлов.")
else:
    # Probe and inspect each candidate before selecting storage. Keep one best
    # path per filesystem, preferring the path with the most reusable files.
    _reserve = 3 * 1024**3
    _candidates = []
    for _base in (Path("/kaggle/temp"), Path("/tmp")):
        try:
            _base.mkdir(parents=True, exist_ok=True)
            with tempfile.NamedTemporaryFile(prefix=".artaug_probe_", dir=str(_base), delete=True):
                pass
            _usage = shutil.disk_usage(_base)
            _st_dev = os.stat(_base).st_dev
            _cache = _base / "artaug_weights"
            _present = 0
            _missing = 0
            for _rel, _rec in WEIGHT_RECORDS.items():
                _p = _cache.joinpath(*_rel.split("/"))
                _size = int(_rec["size_bytes"])
                if _p.is_file() and _p.stat().st_size == _size:
                    _present += _size
                else:
                    _missing += _size
            _candidates.append({"base":_base, "cache":_cache, "free":_usage.free,
                                "device":_st_dev, "present":_present, "missing":_missing})
        except OSError:
            continue

    # Candidate paths can share a mount. Within each device retain its most
    # populated cache (tie-break toward /kaggle/temp), then compare free space.
    _by_device = {}
    for _c in _candidates:
        _old = _by_device.get(_c["device"])
        if _old is None or (_c["present"], _c["base"].as_posix().endswith("/kaggle/temp")) > (_old["present"], _old["base"].as_posix().endswith("/kaggle/temp")):
            _by_device[_c["device"]] = _c
    _candidates = list(_by_device.values())
    _zero_missing = next((c for c in sorted(_candidates, key=lambda x:x["present"], reverse=True) if c["missing"] == 0), None)
    if _zero_missing is not None:
        _chosen = _zero_missing
        WEIGHTS_READY_FOR_LOCAL_LOAD = True
        WEIGHT_SOURCE = "ephemeral_cache"
    else:
        _suitable = [c for c in _candidates if c["free"] >= c["missing"] + _reserve]
        _chosen = max(_suitable, key=lambda x:(x["present"], x["free"])) if _suitable else None
        WEIGHT_SOURCE = "huggingface"
    if _chosen is None:
        _rows = [f"{c['base']}: свободно {c['free']/1024**3:.2f} GiB, пригодный кэш {c['present']/1024**3:.2f} GiB, не хватает {c['missing']/1024**3:.2f} GiB (устройство {c['device']})" for c in _candidates]
        _details = "\n".join(_rows) if _rows else "Не найдено доступного для записи временного хранилища."
        raise RuntimeError("Ни на одном отдельном временном диске не хватает места для весов и резерва 3 GiB.\n"
            "Проверены /kaggle/temp и /tmp; совпадающие файловые системы объединены:\n" + _details + "\n"
            "Подключите полный частный Kaggle Input с папками artaug_weights/kontext, flux_dev и artaug "
            "или выберите среду с большим временным диском. Передача весов не начиналась.")
    LOCAL_WEIGHTS_DIR = _chosen["cache"]
    LOCAL_WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Весовые файлы: {TOTAL_WEIGHT_BYTES/1024**3:.2f} GiB; пригодно в кэше: {_chosen['present']/1024**3:.2f} GiB.")
    print(f"Хранилище: {LOCAL_WEIGHTS_DIR}; свободно: {_chosen['free']/1024**3:.2f} GiB; не хватает: {_chosen['missing']/1024**3:.2f} GiB.")
    if WEIGHTS_READY_FOR_LOCAL_LOAD:
        print("Все файлы уже есть в локальном кэше; авторизация и передача HF не нужны.")
    else:
        print("После проверки доступа Hugging Face можно передавать недостающие файлы.")
    print("Сохраняйте результаты в /kaggle/working.")
WEIGHT_RECORDS = WEIGHTS_MANIFEST["files"]

PREFLIGHT_OK = True

## 4. Check the GPU and install libraries

The first cell runs a small GPU calculation before downloading model weights. The second installs the required libraries and shows installation progress. If it asks you to restart the kernel, restart this session's kernel and then rerun the cells from the beginning. The GPU check confirms that the required operations work; the first photo cell exercises a complete image.

In [ ]:
if not globals().get("PREFLIGHT_OK"):
    raise RuntimeError("Сначала выполните preflight Kaggle.")
import torch
CUDA_SMOKE_OK = False
if not torch.cuda.is_available():
    raise RuntimeError("GPU в Kaggle недоступен. Включите GPU accelerator и перезапустите session.")
if not hasattr(torch, "float8_e4m3fn"):
    raise RuntimeError("Установленный PyTorch не поддерживает float8_e4m3fn.")
props = torch.cuda.get_device_properties(0)
free_bytes, total_bytes = torch.cuda.mem_get_info(0)
_, _ram_free = session_ram()
print(f"Доступная оперативная память с учётом ограничений сессии: {_ram_free/1024**3:.1f} GiB")
if min(free_bytes, _ram_free) < 4 * 1024**3:
    raise RuntimeError("Для пробного запуска освободите хотя бы 4 GiB оперативной памяти и 4 GiB видеопамяти. Полная модель может потребовать больше.")
try:
    a = torch.randn((64, 64), device="cuda", dtype=torch.bfloat16)
    b = a @ a.T
    q = torch.randn((64, 64), device="cuda").to(torch.float8_e4m3fn).float()
    torch.cuda.synchronize()
    if not torch.isfinite(b.float()).all().item() or not torch.isfinite(q).all().item():
        raise RuntimeError("CUDA операция вернула неконечные значения.")
except Exception as exc:
    raise RuntimeError(f"Проверка CUDA BF16/FP8 не пройдена ({type(exc).__name__}).") from None
finally:
    for name in ("a", "b", "q"):
        if name in globals(): del globals()[name]
    torch.cuda.empty_cache()
print(f"GPU: {props.name}; свободная VRAM {free_bytes/1024**3:.1f}/{total_bytes/1024**3:.1f} GiB")
CUDA_SMOKE_OK = True

In [ ]:
if not globals().get("PREFLIGHT_OK") or not globals().get("CUDA_SMOKE_OK"):
    raise RuntimeError("Сначала выполните preflight Kaggle и проверку GPU.")
# Проверка закреплённых пакетов с сохранением CUDA PyTorch
import subprocess, os, sys, time, json, importlib.metadata
from pathlib import Path

from collections import deque

def run_pip_with_progress(command, title):
    print(title, flush=True)
    started = time.monotonic()
    last_lines = deque(maxlen=8)
    env = {**os.environ, 'PYTHONIOENCODING':'utf-8', 'PYTHONUNBUFFERED':'1'}
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, encoding='utf-8', errors='replace', env=env) as process:
        try:
            for line in process.stdout:
                line = line.rstrip()
                last_lines.append(line)
                if line and not line.startswith('Requirement already satisfied:'):
                    print(line, flush=True)
            result = process.wait()
        except BaseException:
            process.terminate()
            process.wait()
            raise
    if result:
        raise RuntimeError('pip остановился. Последние строки:\n'+'\n'.join(last_lines))
    print(f'Этап завершён за {time.monotonic()-started:.0f} с.', flush=True)

pinned = {
    "diffsynth": "2.1.12", "huggingface_hub": "0.36.0", "transformers": "4.57.6",
    "peft": "0.18.1", "accelerate": "1.12.0", "modelscope": "1.34.0", "tokenizers": "0.22.2",
}
helpers = ("sentencepiece", "protobuf", "packaging", "Pillow", "psutil")

def installed_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None

protected = {name: installed_version(name) for name in ("torch", "torchvision")}
if not all(protected.values()):
    raise RuntimeError("В Kaggle включите GPU accelerator; среда должна содержать torch и torchvision.")

needs_install = any(installed_version(name) != version for name, version in pinned.items())
needs_install |= any(installed_version(name) is None for name in helpers)
_import_names = {'diffsynth':'diffsynth','huggingface_hub':'huggingface_hub','transformers':'transformers',
                 'peft':'peft','accelerate':'accelerate','modelscope':'modelscope','tokenizers':'tokenizers'}
restart_needed = any(_import_names[name] in sys.modules and installed_version(name) != version for name,version in pinned.items())
if needs_install:
    constraints = PACKAGE_DIR / "artaug_torch_constraints.txt"
    constraints.write_text("".join(f"{name}=={version}\n" for name, version in protected.items()), encoding="utf-8")
    report = PACKAGE_DIR / "artaug_dependency_plan.json"
    packages = [f"{name}=={version}" for name, version in pinned.items()] + list(helpers)
    pip = [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "--progress-bar", "off", "--constraint", str(constraints)]
    run_pip_with_progress(pip + ["--dry-run", "--report", str(report)] + packages,
                          "Проверяется совместимость библиотек с установленным CUDA PyTorch…")
    for item in json.loads(report.read_text(encoding="utf-8"))["install"]:
        name = item["metadata"]["name"].lower()
        if name in protected and item["metadata"]["version"] != protected[name]:
            raise RuntimeError(f"pip попытался изменить {name}. Проверьте совместимость зависимостей в текущем Python runtime.")
        if name == "diffsynth":
            wheel_sha = item["download_info"].get("archive_info", {}).get("hashes", {}).get("sha256")
            if wheel_sha != "399ed6012da0309982cfdce9e83a293f00b2f20621e4dde195658d54ff49394f":
                raise RuntimeError("Для DiffSynth ожидается проверенный wheel версии 2.1.12.")
    run_pip_with_progress(pip + packages, "Устанавливаются библиотеки…")
    if restart_needed:
        raise RuntimeError("Пакеты установлены, но ядро уже импортировало старые версии. Перезапустите ядро Python в активной сессии из меню Run, затем снова выполните подготовку. Не останавливайте сессию.")

for name, version in {**pinned, **protected}.items():
    if installed_version(name) != version:
        raise RuntimeError(f"Для {name} ожидается версия {version}. Проверьте выбранное ядро.")
print("Библиотеки готовы. PyTorch:", protected["torch"], "· DiffSynth:", pinned["diffsynth"])
try:
    from diffsynth.pipelines.flux_image import FluxImagePipeline, ModelConfig
    from transformers import CLIPTokenizer, T5Tokenizer
    import sentencepiece
except Exception as exc:
    raise RuntimeError(f"Не удалось импортировать DiffSynth Flux, CLIP/T5 tokenizers или SentencePiece ({type(exc).__name__}).") from exc

## 5. Obtain the weights and load ArtAug

Run the four cells in this section in order. They check the token, verify access to the pinned files, show download progress for the 14 files, then verify hashes and load the model. If all weights are already attached, the first cells skip the network download.

If you receive HTTP 403, open both model pages linked at the top while signed in to the account that owns the token. After access is granted, rerun the access check and download. If the network drops, retry the transfer cell in the same session.

In [ ]:
if not globals().get("PREFLIGHT_OK") or not globals().get("CUDA_SMOKE_OK"):
    raise RuntimeError("Сначала выполните preflight Kaggle и проверку GPU.")
HF_WEIGHT_ACCESS_VERIFIED = False
HF_TOKEN_VALUE = None
_LOCAL_READY = bool(globals().get("LOCAL_WEIGHTS_READY")) or bool(globals().get("WEIGHTS_ALREADY_LOCAL")) or bool(globals().get("WEIGHTS_READY_FOR_LOCAL_LOAD"))
if _LOCAL_READY:
    print("Все необходимые файлы уже локальны; проверка токена пропущена.")
else:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN_VALUE = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        HF_TOKEN_VALUE = None
    if not HF_TOKEN_VALUE:
        raise RuntimeError("Не найден Kaggle Secret HF_TOKEN с правом Read. Интерактивный ввод отключён.")
    # Наличие токена явно разрешает новый сетевой запрос даже после offline-загрузки.
    import os
    os.environ.pop("HF_HUB_OFFLINE", None)
    os.environ.pop("TRANSFORMERS_OFFLINE", None)
    import huggingface_hub.constants as _hub_constants
    _hub_constants.HF_HUB_OFFLINE = False
    from huggingface_hub import HfApi
    try:
        _who = HfApi().whoami(token=HF_TOKEN_VALUE)
        _role = ((_who.get("auth") or {}).get("accessToken") or {}).get("role")
        if _role not in ("read", "fineGrained"):
            raise ValueError("token is not read-scoped")
    except Exception as exc:
        HF_TOKEN_VALUE = None
        raise RuntimeError(f"Проверка токена Hugging Face не пройдена ({type(exc).__name__}).") from None
    print("Токен Read принят из Kaggle Secrets.")

In [ ]:
if not globals().get("PREFLIGHT_OK") or not globals().get("CUDA_SMOKE_OK"):
    raise RuntimeError("Сначала выполните preflight Kaggle и проверку GPU.")
HF_WEIGHT_ACCESS_VERIFIED = False
_LOCAL_READY = bool(globals().get("LOCAL_WEIGHTS_READY")) or bool(globals().get("WEIGHTS_ALREADY_LOCAL")) or bool(globals().get("WEIGHTS_READY_FOR_LOCAL_LOAD"))
if _LOCAL_READY:
    print("Локальные веса доступны; metadata запросы Hugging Face пропущены.")
elif not globals().get("HF_TOKEN_VALUE"):
    raise RuntimeError("Сначала выполните проверку Kaggle Secret HF_TOKEN.")
else:
    from huggingface_hub import hf_hub_url
    from huggingface_hub.file_download import get_hf_file_metadata
    _repo_names = {k:v["repo_id"] for k,v in WEIGHTS_MANIFEST["repos"].items()}
    _repo_revs = {k:v["revision"] for k,v in WEIGHTS_MANIFEST["repos"].items()}
    _prefixes = {"kontext":"kontext/", "flux_dev":"flux_dev/", "artaug":"artaug/"}
    for _rel, _rec in sorted(WEIGHTS_MANIFEST["files"].items()):
        _key = next(k for k,p in _prefixes.items() if _rel.startswith(p))
        _filename = _rel[len(_prefixes[_key]):]
        try:
            _info = get_hf_file_metadata(hf_hub_url(_repo_names[_key], _filename, revision=_repo_revs[_key]), token=HF_TOKEN_VALUE, timeout=45)
        except Exception as exc:
            _status = getattr(getattr(exc, "response", None), "status_code", None)
            if _status in (401, 403) or "Gated" in type(exc).__name__ or "RepositoryNotFound" in type(exc).__name__:
                raise RuntimeError(f"Нет доступа к {_repo_names[_key]} (HTTP {_status or '401/403'}). Примите условия FLUX.1-dev и FLUX.1-Kontext-dev на Hugging Face, затем проверьте Kaggle Secret HF_TOKEN с правом Read.") from None
            raise RuntimeError(f"Не удалось проверить метаданные {_repo_names[_key]} ({type(exc).__name__}). Проверьте сеть и повторите ячейку; загрузки ещё не было.") from None
        if int(_info.size) != int(_rec["size_bytes"]) or getattr(_info, "commit_hash", None) != _repo_revs[_key]:
            raise RuntimeError(f"Метаданные {_rel} не совпали с манифестом.")
    HF_WEIGHT_ACCESS_VERIFIED = True
    print("Доступ ко всем закреплённым файлам подтверждён.")

In [ ]:
if not globals().get('CUDA_SMOKE_OK'):
    raise RuntimeError('Сначала выполните проверку GPU.')
if WEIGHTS_READY_FOR_LOCAL_LOAD:
    print('Все файлы уже на месте. Следующая ячейка проверит их контрольные суммы.')
else:
    import shutil, os
    if not HF_WEIGHT_ACCESS_VERIFIED:
        raise RuntimeError("Сначала завершите metadata preflight для всех файлов; передача не начнётся.")
    from huggingface_hub import hf_hub_download
    _repo_names = {"kontext":"black-forest-labs/FLUX.1-Kontext-dev", "flux_dev":"black-forest-labs/FLUX.1-dev", "artaug":"DiffSynth-Studio/FLUX.1-Kontext-dev-lora-ArtAug"}
    _repo_revs = {k:v["revision"] for k,v in WEIGHTS_MANIFEST["repos"].items()}
    _prefixes = {"kontext":"kontext/", "flux_dev":"flux_dev/", "artaug":"artaug/"}
    _records = sorted(WEIGHTS_MANIFEST["files"].items())
    _failures = []
    for _i, (_rel, _rec) in enumerate(_records, 1):
            _key = next(k for k,p in _prefixes.items() if _rel.startswith(p))
            _filename = _rel[len(_prefixes[_key]):]
            _destination = LOCAL_WEIGHTS_DIR.joinpath(*_rel.split("/"))
            _destination.parent.mkdir(parents=True, exist_ok=True)
            if _destination.is_file() and _destination.stat().st_size == int(_rec["size_bytes"]):
                print(f"[{_i}/14] Пропуск уже имеющегося файла правильного размера: {_rel}", flush=True)
                continue
            print(f"[{_i}/14] Получение {_rel} ({int(_rec['size_bytes'])/1024**3:.2f} GiB)...", flush=True)
            try:
                _downloaded = Path(hf_hub_download(
                    repo_id=_repo_names[_key], filename=_filename, revision=_repo_revs[_key],
                    token=HF_TOKEN_VALUE, local_dir=str(LOCAL_WEIGHTS_DIR/_key),
                ))
            except Exception as _exc:
                _status = getattr(getattr(_exc, "response", None), "status_code", None)
                _failures.append((_rel, type(_exc).__name__, _status))
                raise RuntimeError(f"Передача {_rel} не завершилась ({type(_exc).__name__}, HTTP {_status or 'нет'}). Повторите ячейку: завершённые файлы и HF resume cache сохраняются.") from None
            if _downloaded.resolve() != _destination.resolve() or not _destination.is_file() or _destination.stat().st_size != int(_rec["size_bytes"]):
                raise RuntimeError(f"Путь или размер {_rel} расходится с манифестом. Повторите передачу.")
    _missing = [r for r,v in _records if not (LOCAL_WEIGHTS_DIR.joinpath(*r.split("/")).is_file() and LOCAL_WEIGHTS_DIR.joinpath(*r.split("/")).stat().st_size == int(v["size_bytes"]))]
    if _missing:
        raise RuntimeError("Не хватает файлов правильного размера: " + ", ".join(_missing))
    HF_TOKEN_VALUE = None
    WEIGHTS_READY_FOR_LOCAL_LOAD = True
    print("Размеры всех 14 файлов готовы. Контрольные суммы будут проверены загрузчиком перед созданием модели.")

In [ ]:
if not WEIGHTS_READY_FOR_LOCAL_LOAD:
    raise RuntimeError('Сначала завершите загрузку всех файлов. Контрольные суммы всё равно будут сверены при загрузке модели.')
"""Verify and load the local model used by ObjectLab_ArtAug_Local_10.ipynb.

The notebook sets LOCAL_WEIGHTS_DIR and executes this helper in its kernel.
The manifest shipped next to this script identifies the exact 14 files.
"""

from pathlib import Path
import hashlib
import importlib.metadata
import json
import os
import sys
import time


BASE_ID = "black-forest-labs/FLUX.1-Kontext-dev"
AUX_ID = "black-forest-labs/FLUX.1-dev"
ADAPTER_ID = "DiffSynth-Studio/FLUX.1-Kontext-dev-lora-ArtAug"
_SCRIPT_DIR = Path(PACKAGE_DIR).resolve()
_MANIFEST_PATH = _SCRIPT_DIR / "artaug_weights" / "manifest.json"
_MANIFEST_SHA256 = "3f988847f8f6c17741b261b4ae2c32d419230831f399c6a9eda082b8470dd8e1"
LOCAL_WEIGHTS_DIR = Path(globals().get("LOCAL_WEIGHTS_DIR", _SCRIPT_DIR / "artaug_weights")).expanduser()
LOCAL_WEIGHT_FILES_VERIFIED = False


def _sha256_local(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(4 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def _git_blob_sha1(path, size):
    digest = hashlib.sha1()
    digest.update(f"blob {size}\0".encode("ascii"))
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(4 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def _git_lfs_pointer_blob_sha1(content_sha256, content_size):
    pointer = (
        "version https://git-lfs.github.com/spec/v1\n"
        f"oid sha256:{content_sha256}\n"
        f"size {content_size}\n"
    ).encode("ascii")
    return hashlib.sha1(f"blob {len(pointer)}\0".encode("ascii") + pointer).hexdigest()


def _manifest_file_records(manifest):
    files = manifest.get("files")
    if not isinstance(files, dict) or not files:
        raise RuntimeError("artaug_weights/manifest.json must have a non-empty 'files' object.")
    return files


def _verify_local_bundle(root):
    manifest_path = _MANIFEST_PATH
    if not manifest_path.is_file():
        raise RuntimeError(f"Local weights manifest is missing: {manifest_path}")
    manifest_bytes = manifest_path.read_bytes()
    if hashlib.sha256(manifest_bytes).hexdigest() != _MANIFEST_SHA256:
        raise RuntimeError(f"Restore the supplied manifest from the ZIP: {manifest_path}")
    manifest = json.loads(manifest_bytes.decode("utf-8"))
    if manifest.get("schema_version") != 1:
        raise RuntimeError("Local weights manifest schema_version must be 1.")

    expected_repos = {
        "kontext": (BASE_ID, root / "kontext" / "flux1-kontext-dev.safetensors"),
        "flux_dev": (AUX_ID, root / "flux_dev"),
        "artaug": (ADAPTER_ID, root / "artaug" / "model.safetensors"),
    }
    repos = manifest.get("repos")
    if not isinstance(repos, dict):
        raise RuntimeError("Local weights manifest must have a 'repos' object with pinned revisions.")
    revisions = {}
    for key, (expected_id, _) in expected_repos.items():
        record = repos.get(key)
        if not isinstance(record, dict) or record.get("repo_id") != expected_id:
            raise RuntimeError(f"Manifest repo '{key}' must identify {expected_id}.")
        revision = record.get("revision")
        if not isinstance(revision, str) or len(revision) != 40 or any(ch not in "0123456789abcdef" for ch in revision.lower()):
            raise RuntimeError(f"Manifest repo '{key}' must have a pinned 40-character commit revision.")
        revisions[key] = revision

    records = _manifest_file_records(manifest)
    required = {
        "kontext/flux1-kontext-dev.safetensors",
        "flux_dev/text_encoder/model.safetensors",
        "flux_dev/text_encoder_2/model-00001-of-00002.safetensors",
        "flux_dev/text_encoder_2/model-00002-of-00002.safetensors",
        "flux_dev/ae.safetensors",
        "artaug/model.safetensors",
        "flux_dev/tokenizer/merges.txt",
        "flux_dev/tokenizer/special_tokens_map.json",
        "flux_dev/tokenizer/tokenizer_config.json",
        "flux_dev/tokenizer/vocab.json",
        "flux_dev/tokenizer_2/special_tokens_map.json",
        "flux_dev/tokenizer_2/spiece.model",
        "flux_dev/tokenizer_2/tokenizer.json",
        "flux_dev/tokenizer_2/tokenizer_config.json",
    }
    missing_records = sorted(required - set(records))
    if missing_records:
        raise RuntimeError("Manifest omits required model weights: " + ", ".join(missing_records))

    actual_paths = set()
    for path in root.rglob("*"):
        parts = path.relative_to(root).parts
        if len(parts) >= 3 and parts[0] in expected_repos and parts[1:3] == ('.cache', 'huggingface'):
            continue  # HF partial-transfer metadata; it is never passed to the model.
        if path.is_symlink():
            raise RuntimeError(f"Symlinks are not allowed in the local weight bundle: {path.relative_to(root)}")
        if path.name.lower() in {"desktop.ini", "thumbs.db", ".ds_store"}:
            continue
        if path.is_file() and path != root / "manifest.json":
            actual_paths.add(path.relative_to(root).as_posix())
    if actual_paths != set(records):
        missing = sorted(set(records) - actual_paths)
        unlisted = sorted(actual_paths - set(records))
        detail = []
        if missing:
            detail.append("missing files: " + ", ".join(missing[:8]))
        if unlisted:
            detail.append("unlisted files: " + ", ".join(unlisted[:8]))
        raise RuntimeError("Local bundle files do not match manifest (" + "; ".join(detail) + ").")

    weight_files = {
        "kontext/flux1-kontext-dev.safetensors",
        "flux_dev/text_encoder/model.safetensors",
        "flux_dev/text_encoder_2/model-00001-of-00002.safetensors",
        "flux_dev/text_encoder_2/model-00002-of-00002.safetensors",
        "flux_dev/ae.safetensors",
        "artaug/model.safetensors",
    }
    verified = []
    content_hashes = {}
    total_bytes = 0
    for rel in sorted(records):
        print("Checking local file:", rel, flush=True)
        # Reject absolute, parent-traversal, and platform-specific path tricks.
        rel_path = Path(rel)
        if rel_path.is_absolute() or ".." in rel_path.parts or "\\" in rel or ":" in rel:
            raise RuntimeError(f"Unsafe local manifest path: {rel!r}")
        path = root / Path(*rel.split("/"))
        if not path.resolve().is_relative_to(root.resolve()):
            raise RuntimeError(f"Manifest file escapes the bundle root: {rel}")
        record = records[rel]
        if not isinstance(record, dict):
            raise RuntimeError(f"Invalid manifest record for {rel}")
        size = record.get("size_bytes")
        digest = record.get("digest")
        hash_type = record.get("hash_type")
        if not isinstance(size, int) or size < 0 or not isinstance(digest, str):
            raise RuntimeError(f"Manifest record needs size_bytes and digest for {rel}")
        actual_size = path.stat().st_size
        if actual_size != size:
            raise RuntimeError(f"File size mismatch for {rel}: expected {size}, found {actual_size}")
        if rel in weight_files and hash_type not in ("sha256", "git_lfs_pointer_sha1"):
            raise RuntimeError(f"Model weight must use SHA-256 or its pinned Git LFS pointer SHA-1: {rel}")
        if rel not in weight_files and hash_type not in ("sha256", "git_blob_sha1", "git_lfs_pointer_sha1"):
            raise RuntimeError(f"Tokenizer/config file has an unsupported manifest hash type: {rel}")

        if hash_type == "sha256":
            actual_digest = _sha256_local(path)
            content_sha256 = actual_digest
        elif hash_type == "git_blob_sha1":
            actual_digest = _git_blob_sha1(path, actual_size)
            content_sha256 = _sha256_local(path)
        else:
            content_sha256 = _sha256_local(path)
            expected_content_sha256 = record.get("content_sha256")
            if expected_content_sha256 and content_sha256.lower() != expected_content_sha256.lower():
                raise RuntimeError(f"Content SHA-256 mismatch for {rel}")
            pointer_size = record.get("pointer_size")
            pointer = (
                "version https://git-lfs.github.com/spec/v1\n"
                f"oid sha256:{content_sha256}\n"
                f"size {actual_size}\n"
            ).encode("ascii")
            if pointer_size is not None and len(pointer) != pointer_size:
                raise RuntimeError(f"Git LFS pointer size mismatch for {rel}")
            actual_digest = _git_lfs_pointer_blob_sha1(content_sha256, actual_size)
        if actual_digest.lower() != digest.lower():
            raise RuntimeError(f"Checksum mismatch for {rel} ({hash_type}); file may be incomplete or altered.")
        verified.append(rel)
        content_hashes[rel] = content_sha256
        total_bytes += actual_size

    return manifest, revisions, hashlib.sha256(manifest_bytes).hexdigest(), verified, total_bytes, content_hashes


def _record_status(status, stage, **extra):
    if "RUN_STATUS" in globals() and isinstance(RUN_STATUS, dict):
        RUN_STATUS.update({"status": status, "stage": stage, **extra})
        writer = globals().get("write_run_status")
        if callable(writer):
            writer()


def _load_verified_local_artaug():
    global LOCAL_WEIGHT_FILES_VERIFIED, pipe, GENERATOR
    LOCAL_WEIGHT_FILES_VERIFIED = False
    pipe = None
    GENERATOR = None
    root = LOCAL_WEIGHTS_DIR.resolve()
    if not root.is_dir():
        raise RuntimeError(
            f"Local model folder not found: {root}. Copy the prepared 'artaug_weights' folder here; "
            "this loader does not download model files."
        )

    try:
        manifest, revisions, manifest_sha256, verified_files, verified_bytes, content_hashes = _verify_local_bundle(root)
    except Exception as exc:
        _record_status("blocked", "local_manifest_verification", errors=[str(exc)[:2000]])
        raise
    LOCAL_WEIGHT_FILES_VERIFIED = True
    _record_status("ready", "local_files_verified", local_files_verified=len(verified_files),
                   local_files_bytes=verified_bytes, local_manifest_sha256=manifest_sha256,
                   local_content_sha256=content_hashes)
    print(f"Verified {len(verified_files)} local files ({verified_bytes / 1024**3:.2f} GiB); manifest SHA-256 {manifest_sha256}.")
    for rel in verified_files:
        print("  verified:", rel)

    if "--verify-only" in sys.argv[1:]:
        print("Verification-only complete; no torch, DiffSynth model API, GPU, or model weights were loaded.")
        return None, None

    # Force all supported HF/Transformers lookup paths offline before model loading.
    os.environ["HF_HUB_OFFLINE"] = "1"
    os.environ["TRANSFORMERS_OFFLINE"] = "1"
    os.environ["DIFFSYNTH_DOWNLOAD_SOURCE"] = "huggingface"
    os.environ["DIFFSYNTH_MODEL_BASE_PATH"] = str(root)
    # The setup cells may already have imported these packages, so also flip
    # their runtime constants; otherwise environment changes can be too late.
    try:
        import huggingface_hub.constants as hub_constants
        hub_constants.HF_HUB_OFFLINE = True
    except Exception:
        pass
    try:
        import transformers.utils.hub as transformers_hub
        transformers_hub.HF_HUB_OFFLINE = True
    except Exception:
        pass

    import torch
    from diffsynth.pipelines.flux_image import FluxImagePipeline, ModelConfig

    if not torch.cuda.is_available():
        raise RuntimeError("CUDA GPU is unavailable. The local bundle is verified, but this low-VRAM FLUX load requires a CUDA GPU.")
    if not hasattr(torch, "float8_e4m3fn"):
        raise RuntimeError("Installed PyTorch lacks float8_e4m3fn required by this verified low-VRAM loading setup.")
    if importlib.metadata.version("diffsynth") != "2.1.12":
        raise RuntimeError("Use the notebook's pinned DiffSynth 2.1.12 setup before loading local weights.")

    # Validate required names and tokenizer directories before constructing any model objects.
    base_path = root / "kontext" / "flux1-kontext-dev.safetensors"
    aux_path = root / "flux_dev"
    adapter_path = root / "artaug" / "model.safetensors"
    for folder in (aux_path / "tokenizer", aux_path / "tokenizer_2"):
        if not folder.is_dir() or not any(p.is_file() for p in folder.rglob("*")):
            raise RuntimeError(f"Tokenizer files missing from local bundle: {folder}")

    free_bytes, total_vram_bytes = torch.cuda.mem_get_info(0)
    vram_limit = max(1.0, free_bytes / 1024**3 - 0.6)
    disk_vram_config = {
        "offload_dtype": "disk", "offload_device": "disk",
        "onload_dtype": "disk", "onload_device": "disk",
        "preparing_dtype": torch.float8_e4m3fn, "preparing_device": "cuda",
        "computation_dtype": torch.bfloat16, "computation_device": "cuda",
    }
    small_vram_config = {
        "offload_dtype": torch.float8_e4m3fn, "offload_device": "cpu",
        "onload_dtype": torch.float8_e4m3fn, "onload_device": "cpu",
        "preparing_dtype": torch.float8_e4m3fn, "preparing_device": "cuda",
        "computation_dtype": torch.bfloat16, "computation_device": "cuda",
    }
    model_configs = [
        ModelConfig(path=str(base_path), **disk_vram_config),
        ModelConfig(path=str(aux_path / "text_encoder/model.safetensors"), **small_vram_config),
        ModelConfig(path=[
            str(aux_path / "text_encoder_2/model-00001-of-00002.safetensors"),
            str(aux_path / "text_encoder_2/model-00002-of-00002.safetensors"),
        ], **disk_vram_config),
        ModelConfig(path=str(aux_path / "ae.safetensors"), **small_vram_config),
    ]
    _record_status("in_progress", "local_full_model_load", local_files_verified=len(verified_files), manifest_sha256=manifest_sha256)
    try:
        import psutil
        free_host_ram_gib = round(psutil.virtual_memory().available / 1024**3, 2)
    except Exception:
        free_host_ram_gib = None
    print("Loading pinned local FLUX.1-Kontext and ArtAug files with network lookup disabled.",
          {"gpu": torch.cuda.get_device_name(0), "vram_limit_gib": round(vram_limit, 2),
           "free_host_ram_gib": free_host_ram_gib})
    started = time.perf_counter()
    try:
        pipe = FluxImagePipeline.from_pretrained(
            torch_dtype=torch.bfloat16,
            device="cuda",
            model_configs=model_configs,
            tokenizer_1_config=ModelConfig(path=str(aux_path / "tokenizer")),
            tokenizer_2_config=ModelConfig(path=str(aux_path / "tokenizer_2")),
            vram_limit=vram_limit,
        )
        pipe.load_lora(pipe.dit, str(adapter_path), alpha=1)
        _lora_a = sum(len(getattr(module, "lora_A_weights", ())) for module in pipe.dit.modules())
        _lora_b = sum(len(getattr(module, "lora_B_weights", ())) for module in pipe.dit.modules())
        if _lora_a <= 0 or _lora_b <= 0 or _lora_a != _lora_b:
            raise RuntimeError(f"ArtAug LoRA patch validation failed (A={_lora_a}, B={_lora_b}).")
    except Exception as exc:
        _record_status("blocked", "local_model_load", errors=[f"{type(exc).__name__}: {str(exc)[:1800]}"])
        raise RuntimeError(
            f"Local FLUX/ArtAug load failed ({type(exc).__name__}): {str(exc)[:1800]}. "
            "Files were verified; check available RAM/VRAM and the pinned DiffSynth environment. No network fallback was attempted."
        ) from None

    GENERATOR = {
        "method": "released adapter inference; not the original ArtAug training method",
        "synthetic_images": True,
        "base_model_id": BASE_ID,
        "base_revision": revisions["kontext"],
        "aux_model_id": AUX_ID,
        "aux_revision": revisions["flux_dev"],
        "adapter_id": ADAPTER_ID,
        "adapter_revision": revisions["artaug"],
        "adapter_sha256": content_hashes["artaug/model.safetensors"],
        "diffsynth": "2.1.12",
        "torch": torch.__version__,
        "compute_dtype": "bfloat16",
        "offload_storage": "float8_e4m3fn",
        "resolution_max_side": 768,
        "steps": 30,
        "embedded_guidance": 2.5,
        "weight_source": "verified_local_bundle",
        "local_manifest_sha256": manifest_sha256,
        "verified_local_file_count": len(verified_files),
        "offload_strategy": "DiffSynth 2.1.12 disk offload for Kontext DiT and T5; CPU FP8 for CLIP and VAE",
        "lora_A_weight_count": _lora_a,
        "lora_B_weight_count": _lora_b,
    }
    _record_status("ready", "local_model_loaded", generator=GENERATOR,
                   model_load_seconds=round(time.perf_counter() - started, 3))
    print("Loaded the released ArtAug adapter from the verified local bundle.",
          {"model_load_seconds": round(time.perf_counter() - started, 3), "manifest_sha256": manifest_sha256})
    return pipe, GENERATOR


try:
    pipe, GENERATOR = _load_verified_local_artaug()
except Exception as exc:
    pipe, GENERATOR = None, None
    stage = "local_model_load" if LOCAL_WEIGHT_FILES_VERIFIED else "local_manifest_verification"
    _record_status("blocked", stage, errors=[f"{type(exc).__name__}: {str(exc)[:1800]}"])
    raise

## 6. Generate the four selected photos

After the message **Loaded the released ArtAug adapter**, run the helper cell, then run **Photo 1 of 4** through **Photo 4 of 4**. The model encodes the prompt and source image, then runs 30 generation steps and decodes the result. On GPUs with limited memory, weights are loaded from disk as needed, so the first step can take time.

When the **AI-edited · ArtAug** caption appears, compare the object's shape, color, details, and background with the source. Each completed pair is saved to `ArtAug_Pilot_Output.zip`. The recorded run contains four generated pairs. The other six photos in the embedded candidate bundle are not represented by generation sections here. Rerunning a photo cell loads its already-saved pair when available.

In [ ]:
import torch
from PIL import Image, ImageOps
from IPython.display import display
PROMPTS = {
    "bottles": "Improve the lighting and visual clarity of this bottle photograph. Reduce distracting glare while preserving the bottle’s shape, colour, cap and label. Keep the background and camera angle unchanged. Do not add or remove objects.",
    "cups": "Improve the exposure and clarity of this cup. Soften harsh shadows while preserving the cup’s shape, colour, handle and any printed design. Keep the background and camera angle unchanged. Do not add or remove objects.",
    "pens": "Improve the lighting and contrast of this pen photograph so its outline is easier to see. Preserve its shape, colour, tip, cap and visible markings. Keep the background and camera angle unchanged. Do not add or remove objects.",
    "scissors": "Improve the lighting and clarity of this scissors photograph. Reduce glare on the blades while preserving both blades, the handles, their colours and the opening angle. Keep the background unchanged. Do not add or remove objects.",
}
if set(PROMPTS) != set(CLASS_ORDER):
    raise RuntimeError("Prompt/class mismatch")
PILOT_SEED = int(PILOT_SELECTION.get("seed", 4127))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "images").mkdir(exist_ok=True)

def _stable_image_id(row):
    return hashlib.sha256(f"{row['class_name']}\n{row['path']}".encode("utf-8")).hexdigest()[:16]

def _atomic_write_json(path, obj):
    path = Path(path)
    tmp = path.with_name(path.name + ".tmp-" + uuid.uuid4().hex)
    try:
        with tmp.open("x", encoding="utf-8", newline="\n") as stream:
            json.dump(obj, stream, indent=2, ensure_ascii=False)
            stream.write("\n")
            stream.flush()
            os.fsync(stream.fileno())
        if path.exists():
            raise FileExistsError(f"Refusing to replace existing manifest outside atomic checkpoint: {path}")
        os.replace(tmp, path)
    finally:
        if tmp.exists():
            tmp.unlink()

def _save_image_no_replace(image, path, **save_kwargs):
    path = Path(path)
    if path.exists():
        raise FileExistsError(f"Refusing to overwrite an existing result image: {path}")
    tmp = path.with_name(path.stem + ".tmp-" + uuid.uuid4().hex + path.suffix)
    try:
        image.save(tmp, **save_kwargs)
        if path.exists():
            raise FileExistsError(f"Result appeared while saving; refusing to overwrite: {path}")
        os.replace(tmp, path)
    finally:
        if tmp.exists():
            tmp.unlink()

def _load_bundle():
    path = OUTPUT_DIR / "manifest.json"
    if not path.exists():
        return {"schema_version": 1, "generator": GENERATOR, "source_manifest_sha256": SOURCE_MANIFEST_SHA256, "examples": []}
    bundle = json.loads(path.read_text(encoding="utf-8"))
    if bundle.get("schema_version") != 1 or bundle.get("source_manifest_sha256") != SOURCE_MANIFEST_SHA256 or bundle.get("generator") != GENERATOR:
        raise RuntimeError("Папка artaug_pilot содержит результаты другой модели/версии/источника; смешивание запрещено.")
    if not isinstance(bundle.get("examples"), list):
        raise RuntimeError("Неверная схема artaug_pilot/manifest.json")
    rows_by_id = {_stable_image_id(row): row for row in SELECTED}
    seen_ids = set()
    for entry in bundle["examples"]:
        image_id = entry.get("id")
        if image_id in seen_ids or image_id not in rows_by_id:
            raise RuntimeError("Manifest содержит повторный или невыбранный ID.")
        seen_ids.add(image_id)
        _validate_existing_entry(entry, rows_by_id[image_id])
    return bundle

def _validate_existing_entry(entry, row):
    image_id = _stable_image_id(row)
    if entry.get("id") != image_id or entry.get("class_name") != row["class_name"] or entry.get("source_manifest_path") != row["path"] or entry.get("source_sha256") != row["sha256"] or entry.get("approved") is not False:
        raise RuntimeError("Сохранённая запись не связана с выбранным train-источником или уже отмечена approved.")
    for field in ("original", "enhanced"):
        rel = Path(entry[field])
        if rel.is_absolute() or ".." in rel.parts or len(rel.parts) != 2 or rel.parts[0] != "images":
            raise RuntimeError("Небезопасный путь результата в manifest")
        path = OUTPUT_DIR / rel
        if not path.is_file() or sha256_file(path) != entry.get(field + "_sha256"):
            raise RuntimeError(f"Файл результата отсутствует или не совпал с SHA-256: {rel}")
    if entry.get("original_sha256") != row["sha256"]:
        raise RuntimeError("Сохранённый original не совпадает побайтно с источником.")

def _atomic_copy_source(source_path, destination):
    if destination.exists():
        raise FileExistsError(f"Refusing to overwrite an existing result image: {destination}")
    tmp = destination.with_name(destination.name + ".tmp-" + uuid.uuid4().hex)
    try:
        with open(source_path, "rb") as source_stream, tmp.open("xb") as target_stream:
            shutil.copyfileobj(source_stream, target_stream, length=1024 * 1024)
            target_stream.flush()
            os.fsync(target_stream.fileno())
        if destination.exists():
            raise FileExistsError(f"Result appeared while saving; refusing to overwrite: {destination}")
        os.replace(tmp, destination)
    finally:
        if tmp.exists():
            tmp.unlink()

def _cleanup_transaction(tx_dir):
    if not tx_dir.exists():
        return
    allowed = {"entry.json", "entry.tmp.json", "original.jpg", "enhanced.jpg"}
    children = list(tx_dir.iterdir())
    import re
    temp_pattern = r"(?:original\.jpg\.tmp-[0-9a-f]{32}|enhanced\.tmp-[0-9a-f]{32}\.jpg)"
    if any((p.name not in allowed and re.fullmatch(temp_pattern, p.name) is None) or not p.is_file() for p in children):
        raise RuntimeError(f"Transaction folder has unexpected files and will not be removed: {tx_dir}")
    for child in children:
        child.unlink()
    tx_dir.rmdir()

def _commit_transaction(entry, row, tx_dir, bundle):
    image_id = _stable_image_id(row)
    _validate_entry_metadata(entry, row)
    for key, staged_name in (("original", "original.jpg"), ("enhanced", "enhanced.jpg")):
        final_path = OUTPUT_DIR / entry[key]
        staged_path = tx_dir / staged_name
        expected = entry[key + "_sha256"]
        if final_path.exists():
            if sha256_file(final_path) != expected:
                raise RuntimeError(f"Existing result differs from transaction and will not be overwritten: {final_path}")
            if staged_path.exists():
                if sha256_file(staged_path) != expected:
                    raise RuntimeError(f"Staged and committed result differ: {staged_path}")
                staged_path.unlink()
        else:
            if not staged_path.is_file() or sha256_file(staged_path) != expected:
                raise RuntimeError(f"Transaction lacks a valid {key} image for {image_id}")
            os.replace(staged_path, final_path)
    _validate_existing_entry(entry, row)
    current = _load_bundle()
    matching = [e for e in current["examples"] if e.get("id") == image_id]
    if matching:
        if len(matching) != 1 or matching[0] != entry:
            raise RuntimeError("A conflicting entry for this image already exists.")
    else:
        current["examples"].append(entry)
    manifest_path = OUTPUT_DIR / "manifest.json"
    tmp_manifest = manifest_path.with_name(manifest_path.name + ".tmp-" + uuid.uuid4().hex)
    try:
        with tmp_manifest.open("x", encoding="utf-8", newline="\n") as stream:
            json.dump(current, stream, indent=2, ensure_ascii=False)
            stream.write("\n")
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(tmp_manifest, manifest_path)
    finally:
        if tmp_manifest.exists():
            tmp_manifest.unlink()
    _cleanup_transaction(tx_dir)
    RUN_STATUS.update({"status": "complete" if len(current["examples"]) == 10 else "ready", "stage": "photo_saved", "generated_count": len(current["examples"]), "total_selected": 10, "errors": []})
    write_run_status()
    export_current()
    return entry, current

def _validate_entry_metadata(entry, row):
    image_id = _stable_image_id(row)
    if entry.get("id") != image_id or entry.get("class_name") != row["class_name"] or entry.get("source_manifest_path") != row["path"] or entry.get("source_sha256") != row["sha256"] or entry.get("original_sha256") != row["sha256"] or entry.get("approved") is not False:
        raise RuntimeError("Транзакция не связана с выбранным train-источником.")
    if entry.get("prompt") != PROMPTS[row["class_name"]] or entry.get("seed") != PILOT_SEED + SELECTED.index(row):
        raise RuntimeError("Промпт/seed транзакции отличается от утверждённой конфигурации.")
    for key, expected in (("original", f"images/{image_id}_original.jpg"), ("enhanced", f"images/{image_id}_enhanced.jpg")):
        if entry.get(key) != expected:
            raise RuntimeError("Небезопасный или неожиданный путь в транзакции.")

def export_current():
    manifest_path = OUTPUT_DIR / "manifest.json"
    if not manifest_path.is_file():
        print("Нет завершённых пар; ZIP пока не создан.")
        return None
    bundle = _load_bundle()
    if STATUS_PATH.is_file():
        status_target = OUTPUT_DIR / "run_status.json"
        tmp_status = status_target.with_name(status_target.name + ".tmp-" + uuid.uuid4().hex)
        try:
            shutil.copyfile(STATUS_PATH, tmp_status)
            os.replace(tmp_status, status_target)
        finally:
            if tmp_status.exists():
                tmp_status.unlink()
    tmp_zip = EXPORT_PATH.with_name(EXPORT_PATH.stem + ".tmp-" + uuid.uuid4().hex + EXPORT_PATH.suffix)
    try:
        with zipfile.ZipFile(tmp_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
            archive.write(manifest_path, manifest_path.relative_to(PACKAGE_DIR))
            referenced = {"run_status.json"}
            for entry in bundle["examples"]:
                referenced.add(entry["original"])
                referenced.add(entry["enhanced"])
            for rel in sorted(referenced):
                path = OUTPUT_DIR / rel
                if path.is_file():
                    archive.write(path, path.relative_to(PACKAGE_DIR))
        os.replace(tmp_zip, EXPORT_PATH)
    finally:
        if tmp_zip.exists():
            tmp_zip.unlink()
    print("Обновлён ZIP:", EXPORT_PATH, "entries:", len(bundle["examples"]))
    return EXPORT_PATH

def _generate_selected_photo(index):
    if pipe is None or GENERATOR is None:
        raise RuntimeError("Сначала выполните ячейку локальной загрузки модели.")
    if not isinstance(index, int) or not 0 <= index < len(SELECTED):
        raise IndexError("Фотография вне диапазона 0..9")
    row = SELECTED[index]
    source_path = INPUT_DIR.joinpath(*Path(row["path"]).parts)
    if sha256_file(source_path) != row["sha256"]:
        raise RuntimeError("SHA-256 источника изменился; генерация остановлена.")
    with Image.open(source_path) as image:
        original = ImageOps.exif_transpose(image).convert("RGB")
    print(f"Исходник {index+1}/10 — {row['class_name']} / {source_path.name}; AI-изображение будет синтетическим.")
    original_preview = original.copy()
    original_preview.thumbnail((600, 600), Image.Resampling.LANCZOS)
    display(original_preview)
    bundle = _load_bundle()
    image_id = _stable_image_id(row)
    previous = [e for e in bundle["examples"] if e.get("id") == image_id]
    if previous:
        if len(previous) != 1:
            raise RuntimeError("В manifest несколько одинаковых ID; ручная проверка обязательна.")
        _validate_existing_entry(previous[0], row)
        from IPython.display import display as _display
        with Image.open(OUTPUT_DIR / previous[0]["original"]) as im_o, Image.open(OUTPUT_DIR / previous[0]["enhanced"]) as im_e:
            print("Этот кадр уже сохранён; повторная генерация пропущена.")
            preview_o, preview_e = im_o.copy(), im_e.copy()
            preview_o.thumbnail((600, 600), Image.Resampling.LANCZOS)
            preview_e.thumbnail((600, 600), Image.Resampling.LANCZOS)
            print("Оригинал")
            _display(preview_o)
            print("AI-edited · ArtAug")
            _display(preview_e)
        export_current()
        return previous[0]

    tx_dir = OUTPUT_DIR / ".transactions" / image_id
    if tx_dir.exists():
        pending_path = tx_dir / "entry.json"
        if pending_path.is_file():
            pending = json.loads(pending_path.read_text(encoding="utf-8"))
            if pending.get("generator") != GENERATOR or pending.get("source_manifest_sha256") != SOURCE_MANIFEST_SHA256:
                raise RuntimeError("Версия незавершённой пары отличается от текущего запуска.")
            pending = pending["example"]
            _validate_entry_metadata(pending, row)
            committed, _ = _commit_transaction(pending, row, tx_dir, bundle)
            with Image.open(OUTPUT_DIR / committed["original"]) as im_o, Image.open(OUTPUT_DIR / committed["enhanced"]) as im_e:
                po, pe = im_o.copy(), im_e.copy()
                po.thumbnail((600, 600), Image.Resampling.LANCZOS)
                pe.thumbnail((600, 600), Image.Resampling.LANCZOS)
                print("Восстановлена завершённая транзакция; повторная генерация не выполнялась.")
                display(po, pe)
            return committed
        _cleanup_transaction(tx_dir)

    original_rel = f"images/{image_id}_original.jpg"
    enhanced_rel = f"images/{image_id}_enhanced.jpg"
    original_out = OUTPUT_DIR / original_rel
    enhanced_out = OUTPUT_DIR / enhanced_rel
    if original_out.exists() or enhanced_out.exists():
        raise RuntimeError("Найдены файлы без manifest-записи. Они не будут перезаписаны; проверьте папку artaug_pilot/images.")
    scale = min(768 / max(original.size), 1.0)
    width = max(16, int(original.width * scale) // 16 * 16)
    height = max(16, int(original.height * scale) // 16 * 16)
    conditioning = original.resize((width, height), Image.Resampling.LANCZOS)
    prompt = PROMPTS[row["class_name"]]
    seed = PILOT_SEED + index
    started_utc = datetime.now(timezone.utc).isoformat()
    started = time.perf_counter()
    def show_steps(timesteps):
        total = len(timesteps)
        print(f'Начинается шаг 1/{total}.', flush=True)
        for completed, timestep in enumerate(timesteps, 1):
            yield timestep
            torch.cuda.synchronize()
            elapsed_steps = time.perf_counter() - started
            print(f'Завершено {completed}/{total} · с начала обработки {elapsed_steps/60:.1f} мин.', flush=True)
        print('Все шаги готовы. Идёт декодирование изображения…', flush=True)

    _record_status('in_progress', 'single_image_generation', current_photo=index+1, errors=[])
    print('Кодирование промпта и фото, подготовка модели к первому шагу…', flush=True)
    try:
        edited = pipe(prompt=prompt, kontext_images=conditioning, height=height, width=width,
                      num_inference_steps=30, embedded_guidance=2.5, seed=seed, progress_bar_cmd=show_steps)
    except Exception as exc:
        message = str(exc)[:1800]
        try:
            RUN_STATUS.update({"status": "blocked", "stage": "single_image_generation", "errors": [message]})
            write_run_status()
        except Exception:
            pass
        raise RuntimeError(f"Генерация кадра {index+1} завершилась ошибкой {type(exc).__name__}: {message}. Замена модели не выполнялась.") from None
    elapsed = time.perf_counter() - started

    # Stage the complete pair before committing either public output file.
    tx_dir.parent.mkdir(exist_ok=True)
    tx_dir.mkdir()
    staged_original = tx_dir / "original.jpg"
    staged_enhanced = tx_dir / "enhanced.jpg"
    _atomic_copy_source(source_path, staged_original)
    _save_image_no_replace(edited, staged_enhanced, format="JPEG", quality=95, optimize=True)
    entry = {
        "id": image_id,
        "class_name": row["class_name"],
        "source_manifest_path": row["path"],
        "source_sha256": row["sha256"],
        "original": original_rel,
        "original_sha256": sha256_file(staged_original),
        "enhanced": enhanced_rel,
        "enhanced_sha256": sha256_file(staged_enhanced),
        "prompt": prompt,
        "seed": seed,
        "approved": False,
        "generation": {
            "started_utc": started_utc,
            "duration_seconds": round(elapsed, 3),
            "gpu": torch.cuda.get_device_name(0),
            "requested_resolution": [width, height],
            "actual_resolution": list(edited.size),
            "conditioning_resolution": [width, height],
            "conditioning_resize": "EXIF-transposed RGB resized with Lanczos; max side 768; dimensions floored to multiples of 16, minimum 16",
            "conditioning_tiled": False,
            "steps": 30,
            "embedded_guidance": 2.5,
        },
    }
    entry_tmp = tx_dir / "entry.tmp.json"
    with entry_tmp.open("x", encoding="utf-8", newline="\n") as stream:
        json.dump({"generator": GENERATOR, "source_manifest_sha256": SOURCE_MANIFEST_SHA256, "example": entry}, stream, indent=2, ensure_ascii=False)
        stream.write("\n")
        stream.flush()
        os.fsync(stream.fileno())
    os.replace(entry_tmp, tx_dir / "entry.json")
    entry, bundle = _commit_transaction(entry, row, tx_dir, bundle)
    from IPython.display import display as _display
    print(f"Saved {row['class_name']} / {source_path.name}; {elapsed:.1f}s; approved=false; seed={seed}.")
    edited_preview = edited.copy()
    edited_preview.thumbnail((600, 600), Image.Resampling.LANCZOS)
    print("Оригинал")
    _display(original_preview)
    print("AI-edited · ArtAug")
    _display(edited_preview)
    return entry

print("Готово. Выберите одну ячейку с фотографией ниже.")
export_current()  # Refresh resumable ZIP if a previous run already committed entries.


def generate_one(index):
    # A second Jupyter kernel using the same folder cannot race a live run.
    with (OUTPUT_DIR / ".generation.lock").open("a+b") as guard:
        guard.seek(0, os.SEEK_END)
        if guard.tell() == 0:
            guard.write(b"0")
            guard.flush()
        guard.seek(0)
        try:
            if os.name == "nt":
                import msvcrt
                msvcrt.locking(guard.fileno(), msvcrt.LK_NBLCK, 1)
            else:
                import fcntl
                fcntl.flock(guard.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
        except OSError as exc:
            raise RuntimeError("Другая ячейка уже работает с этой папкой. Дождитесь её завершения.") from exc
        return _generate_selected_photo(index)

### Photo 1 of 4: Bottles

Source: `train/bottles/photo_5426963673156099989_y.jpg`

In [ ]:
generate_one(0)

### Photo 2 of 4: Cups

Source: `train/cups/6.jpg`

In [ ]:
generate_one(1)

### Photo 3 of 4: Pens

Source: `train/pens/78.jpg`

In [ ]:
generate_one(2)

### Photo 4 of 4: Scissors

Source: `train/scissors/62.jpg`

In [ ]:
generate_one(3)

## 7. Download the four generated pairs

In Kaggle, open **Output / Files**, refresh the file list, and download **ArtAug_Pilot_Output.zip**. The archive is updated after each generated photo; download the latest copy before ending the session. To continue in a later session, attach this output as an input to the new run. To keep the executed notebook, save the current version with outputs, download the `.ipynb` through **File → Download Notebook**, and confirm that the four source/edit pairs are visible.

Label edited images as **AI-edited · ArtAug** in the app and report. Review every output before use. Obtain consent before publishing images that show identifiable people.

In [ ]:
if EXPORT_PATH.is_file():
    manifest, rows, _ = _verified_zip(EXPORT_PATH)
    print(f'Готовых пар: {len(rows)}/10. Скачайте файл через панель Output:')
    print(EXPORT_PATH)
else:
    print('Архив появится после первого успешно сохранённого фото.')